In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

In [3]:
import numpy as np
import torch

In [4]:
import bayesgpt

In [5]:
from bayesgpt.networks import BayesGPT, BayesGPTv1

In [6]:
from bayesgpt.simulators.model_family import NestedModelFamily
from bayesgpt.simulators.benchmarks.ddms import DDM
from bayesgpt.simulators.benchmarks.ddms.ddm_priors import ddm_baseline_priors
from bayesgpt.adapters import Adapter

In [7]:
ddm_family = NestedModelFamily(
    name="ddm",
    model=DDM(),
    prior_fun=ddm_baseline_priors(),
    regressed_params=["v", "a", "tau"],
    mask_randomizer_kwargs=dict(
        free_intrinsics=["v", "a", "tau"],
        fixed_intrinsics=["s_v", "s_tau"],
        fixed_values={"s_v": 0, "s_tau": 0},
    )
)

In [8]:
sample_kwargs = {
    'min_num_regressors': 2,
    "max_num_regressors": 2,
    "max_num_categories": 2,
    "fixed_config": True
}

samples = ddm_family.batch_sample(
    batch_size=20,
    num_obs=50,
    flatten_param_outputs=True,
    **sample_kwargs
)

adapter = Adapter()

/home/radevs/anaconda3/envs/bf/lib/python3.11/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [9]:
for k, v in samples.items():
    print(k, v.shape if isinstance(v, np.ndarray) else v)

model_names ['ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm', 'ddm']
design_configs [{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v', 'a', 'tau']}, {'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['v',

In [10]:
# Adapt
adapted = adapter.adapt(samples, intrinsic_params=ddm_family.intrinsic_params)

In [11]:
adapted["param_indices"].shape

torch.Size([20, 15, 1])

In [12]:
p = adapted["param_matrices"][..., None]

In [13]:
p.device

device(type='cuda', index=0)

In [14]:
bayesgpt = BayesGPT(encoder_input_dim=5, encoder_num_layers=4, decoder_num_layers=4, seed_dim=64, num_seeds=10)
# bayesgpt = BayesGPTv1(encoder_input_dim=5, encoder_num_layers=4, decoder_num_layers=4, seed_dim=64, num_seeds=10)
bayesgpt = bayesgpt.to("cuda")

In [15]:
print(adapted['input_data'].shape)
print(p.shape)

torch.Size([20, 50, 5])
torch.Size([20, 15, 1])


In [16]:
adapted['param_indices'].shape

torch.Size([20, 15, 1])

In [30]:
pred_velocity, target_velocity = bayesgpt(
    p,
    adapted['input_data'],
    adapted['param_indices'],
    adapted['regressor_indices'],
    adapted['param_masks']
)

loss = bayesgpt.compute_loss(pred_velocity, target_velocity, adapted['param_masks'])

print(loss)

samples = bayesgpt.sample(
    adapted['input_data'],
    adapted['param_indices'],
    adapted['regressor_indices'],
    adapted['param_masks'],
    steps=100,
    num_samples=500
)

print(samples.shape)

tensor(2.3156, device='cuda:0', grad_fn=<MeanBackward0>)


Sampling:   0%|          | 0/500 [00:00<?, ?sample/s]

(20, 500, 15, 1)


In [23]:
adapted["param_matrices"][0]

tensor([ 1.1159, -0.5076, -1.5490,  0.0000,  0.0000, -0.4911,  0.5747, -0.0158,
         0.0000,  0.0000,  0.4556, -1.0298, -2.7534,  0.0000,  0.0000],
       device='cuda:0')

In [24]:
adapted["param_matrices"][1]

tensor([ 2.8594, -0.1039, -1.6858,  0.0000,  0.0000, -0.4901, -1.9129,  0.3153,
         0.0000,  0.0000, -0.0440, -0.6402, -0.0074,  0.0000,  0.0000],
       device='cuda:0')

In [25]:
adapted["param_matrices"][2]

tensor([ 1.3612,  0.0618, -1.2231,  0.0000,  0.0000, -0.6551, -1.1731,  1.8512,
         0.0000,  0.0000, -1.0814, -2.1571, -2.0337,  0.0000,  0.0000],
       device='cuda:0')

In [26]:
adapted["param_masks"][2]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1., 0., 0.],
       device='cuda:0')